# IMD rainfall

Two parts:
1. download the daily 0.25° grids from IMD
2. turn those grids into district totals (Kharif / Rabi / Annual)

Crop table uses years like `1997-1998`, so I keep the same window: 1997 to 2023.

In [1]:
from pathlib import Path
import json
import re
from collections import defaultdict

import imdlib as imd
import numpy as np
import pandas as pd
import xarray as xr
from shapely.geometry import Point, shape
from shapely.strtree import STRtree

In [2]:
start_year = 1997
end_year = 2023

# imdlib wants a main folder that contains a rain/ subfolder
data_dir = Path("data/imd")
rain_dir = data_dir / "rain"
rain_dir.mkdir(parents=True, exist_ok=True)

out_csv = Path("data/processed/rainfall_district_seasonal.csv")
out_csv.parent.mkdir(parents=True, exist_ok=True)

## 1. Download

`imd.get_data` pulls the binary `.grd` files from IMD Pune. I rename them yearwise so each file is just `1997.grd`, `1998.grd`, …

No point downloading a year twice, so I check disk first.

In [3]:
already_have = []
need = []

for year in range(start_year, end_year + 1):
    f = rain_dir / f"{year}.grd"
    if f.exists():
        already_have.append(year)
    else:
        need.append(year)

print("already downloaded:", len(already_have))
print("missing:", need)

already downloaded: 27
missing: []


In [4]:
if len(need) == 0:
    print("nothing to download")
else:
    # get_data takes a start year and an end year, not a list
    first = min(need)
    last = max(need)
    print("downloading", first, "to", last)
    imd.get_data(
        "rain",                 # rainfall, not temperature
        first,
        last,
        fn_format="yearwise",   # save as 1997.grd instead of the IMD default name
        file_dir=str(data_dir),
    )

print("files on disk:", len(list(rain_dir.glob("*.grd"))))

nothing to download
files on disk: 27


In [5]:
raw = imd.open_data("rain", 1997, 1997, "yearwise", str(data_dir))
ds = raw.get_xarray()
print(ds)
print("value range including fill:", float(ds["rain"].min()), float(ds["rain"].max()))

ds = ds.where(ds["rain"] != -999.0)
print("after dropping -999, max is", float(ds["rain"].max()))

<xarray.Dataset> Size: 51MB
Dimensions:  (time: 365, lat: 129, lon: 135)
Coordinates:
  * lat      (lat) float64 1kB 6.5 6.75 7.0 7.25 7.5 ... 37.75 38.0 38.25 38.5
  * lon      (lon) float64 1kB 66.5 66.75 67.0 67.25 ... 99.25 99.5 99.75 100.0
  * time     (time) datetime64[ns] 3kB 1997-01-01 1997-01-02 ... 1997-12-31
Data variables:
    rain     (time, lat, lon) float64 51MB -999.0 -999.0 ... -999.0 -999.0
Attributes:
    Conventions:  CF-1.7
    title:        IMD gridded data
    source:       https://imdpune.gov.in/
    history:      2026-09-06 05:54:46.015717 Python
    references:   
    comment:      
    crs:          epsg:4326
value range including fill: -999.0 425.4890441894531
after dropping -999, max is 425.4890441894531


## 2. District totals

Each grid cell is about 25 km. I put the cell on a district if the **centre** of the cell falls inside the polygon.

A few districts are smaller than one cell, so they never catch a centre. For those I take the nearest cell to the district centroid — otherwise they would stay grey on the rainfall map for no good reason.

Seasons (same labels as the crop file):
- Kharif = 1 Jun to 30 Sep
- Rabi = 1 Oct to 31 Mar (tagged to the year it started)
- Annual = 1 Jan to 31 Dec of the start year

If a district-season has less than 90% of expected days, I drop it. I don't scale a half-year up to look complete.

In [6]:
def normalize(name):
    # same join key as the crop cleaning / map
    text = str(name or "").lower().strip().replace("&", "and")
    text = re.sub(r"[^a-z0-9]+", "", text)
    if text.endswith("district") and len(text) > len("district"):
        text = text[:-len("district")]
    return text


def agri_year(y):
    # crop csv is 1997-1998, so keep that here
    return f"{y}-{y + 1}"

In [7]:
with open("data/INDIA_DISTRICTS.geojson", encoding="utf-8") as f:
    geo = json.load(f)

geoms = []
info = []

for feat in geo["features"]:
    props = feat.get("properties") or {}
    name = props.get("district") or props.get("DISTRICT") or ""
    geom = feat.get("geometry")
    if not name or not geom:
        continue
    poly = shape(geom)
    if poly.is_empty:
        continue
    geoms.append(poly)
    info.append({
        "state": str(props.get("state") or props.get("STATE") or "").strip(),
        "district": str(name).strip(),
        "key": normalize(name),
    })

tree = STRtree(geoms)
print(len(geoms), "districts")

792 districts


In [8]:
def open_year(year):
    raw = imd.open_data("rain", year, year, "yearwise", str(data_dir))
    ds = raw.get_xarray()
    return ds.where(ds["rain"] != -999.0)


def map_cells_to_districts(ds):
    # land cells = anything that is not all-NaN over the year
    land = ds["rain"].notnull().any("time")
    lat2d, lon2d = xr.broadcast(ds.lat, ds.lon)
    lats = lat2d.values[land.values]
    lons = lon2d.values[land.values]

    rows = []
    seen = set()
    for lat, lon in zip(lats, lons):
        pt = Point(float(lon), float(lat))
        hits = tree.query(pt, predicate="contains")
        if len(hits) == 0:
            hits = tree.query(pt, predicate="intersects")
        if len(hits) == 0:
            continue
        rec = info[int(hits[0])]
        seen.add(rec["key"])
        rows.append({
            "lat": float(lat), "lon": float(lon),
            "state": rec["state"], "district": rec["district"], "key": rec["key"],
        })

    mapping = pd.DataFrame(rows)

    # tiny districts that missed every cell centre
    rain0 = ds["rain"].isel(time=0)
    extra = []
    for i, rec in enumerate(info):
        if rec["key"] in seen:
            continue
        c = geoms[i].centroid
        cell = rain0.sel(lat=float(c.y), lon=float(c.x), method="nearest")
        extra.append({
            "lat": float(cell.lat), "lon": float(cell.lon),
            "state": rec["state"], "district": rec["district"], "key": rec["key"],
        })
    if extra:
        print("nearest-cell fallback for", len(extra), "districts")
        mapping = pd.concat([mapping, pd.DataFrame(extra)], ignore_index=True)

    print(len(mapping), "cells ->", mapping["key"].nunique(), "districts")
    return mapping

In [9]:
def which_seasons(stamp):
    # one calendar day can feed Annual and also Kharif or Rabi
    month = int(stamp.month)
    year = int(stamp.year)
    out = [("Annual", year)]
    if month in (6, 7, 8, 9):
        out.append(("Kharif", year))
    elif month in (10, 11, 12):
        out.append(("Rabi", year))
    elif month in (1, 2, 3):
        # Jan-Mar belongs to the Rabi that started last October
        out.append(("Rabi", year - 1))
    return out

In [10]:
rebuild = False

if out_csv.exists() and not rebuild:
    rain = pd.read_csv(out_csv)
    print("loaded existing", out_csv, rain.shape)
else:
    print("mapping cells using 1997 grid")
    sample = open_year(1997)
    mapping = map_cells_to_districts(sample)
    del sample

    totals = defaultdict(lambda: {"rain": 0.0, "n": 0})

    lat_da = xr.DataArray(mapping["lat"].to_numpy(), dims="cell")
    lon_da = xr.DataArray(mapping["lon"].to_numpy(), dims="cell")
    keys = mapping["key"].to_numpy()
    states = mapping["state"].to_numpy()
    districts = mapping["district"].to_numpy()
    uniq, inverse = np.unique(keys, return_inverse=True)
    first = {k: int(np.flatnonzero(keys == k)[0]) for k in uniq}

    for year in range(start_year, end_year + 1):
        print(" ", year)
        ds = open_year(year)
        grid = ds["rain"].sel(lat=lat_da, lon=lon_da).values
        times = pd.to_datetime(ds["time"].values)

        for t, stamp in enumerate(times):
            day = grid[t]
            ok = np.isfinite(day)
            if not ok.any():
                continue
            sums = np.zeros(len(uniq))
            counts = np.zeros(len(uniq), dtype=np.int32)
            np.add.at(sums, inverse[ok], day[ok])
            np.add.at(counts, inverse[ok], 1)
            for season, crop_year in which_seasons(stamp):
                for i, key in enumerate(uniq):
                    if counts[i] == 0:
                        continue
                    row_i = first[key]
                    slot = totals[(states[row_i], districts[row_i], key, crop_year, season)]
                    slot["rain"] += float(sums[i] / counts[i])
                    slot["n"] += 1
        del ds

    expected = {"Kharif": 122, "Rabi": 182, "Annual": 365}
    rows = []
    for (state, district, key, crop_year, season), slot in totals.items():
        need_days = expected[season]
        if season == "Annual" and crop_year % 4 == 0:
            need_days = 366
        if slot["n"] / need_days < 0.90:
            continue
        rows.append({
            "state": state,
            "district": district,
            "key": key,
            "year": agri_year(int(crop_year)),
            "season": season,
            "rain_mm": slot["rain"],
            "n_days": slot["n"],
        })

    rain = pd.DataFrame(rows)
    rain["rain_normal_mm"] = rain.groupby(["key", "season"])["rain_mm"].transform("mean")
    rain["rain_anom_mm"] = rain["rain_mm"] - rain["rain_normal_mm"]
    rain["rain_anom_pct"] = np.where(
        rain["rain_normal_mm"] != 0,
        100.0 * rain["rain_anom_mm"] / rain["rain_normal_mm"],
        np.nan,
    )
    rain = rain.round(2)
    rain.to_csv(out_csv, index=False)
    print("wrote", len(rain), "->", out_csv)

rain.head()

loaded existing data\processed\rainfall_district_seasonal.csv (62640, 10)


,state,district,key,year,season,rain_mm,n_days,rain_normal_mm,rain_anom_mm,rain_anom_pct
0,WEST BENGAL,>L|PUR DU>R,lpurdur,1997-1998,Annual,2965.16,365,3946.17,-981.01,-24.86
1,WEST BENGAL,>L|PUR DU>R,lpurdur,1998-1999,Annual,5027.17,365,3946.17,1080.99,27.39
2,WEST BENGAL,>L|PUR DU>R,lpurdur,1999-2000,Annual,4190.37,365,3946.17,244.20,6.19
3,WEST BENGAL,>L|PUR DU>R,lpurdur,2000-2001,Annual,4267.22,366,3946.17,321.05,8.14
4,WEST BENGAL,>L|PUR DU>R,lpurdur,2001-2002,Annual,3369.98,365,3946.17,-576.19,-14.60


In [11]:
print(rain["season"].value_counts())

# new district on the current map, old years still get rain because we used today's polygons
t = rain[rain["district"].str.contains("Tirupati", case=False, na=False)]
t[t["season"] == "Annual"][["year", "rain_mm", "rain_anom_pct"]].head(8)

season
Annual    21141
Kharif    21141
Rabi      20358
Name: count, dtype: int64


,year,rain_mm,rain_anom_pct
57680,1997-1998,1297.03,12.46
57681,1998-1999,1091.96,-5.32
57682,1999-2000,739.45,-35.89
57683,2000-2001,825.03,-28.47
57684,2001-2002,1094.82,-5.07
57685,2002-2003,963.20,-16.49
57686,2003-2004,840.11,-27.16
57687,2004-2005,1054.84,-8.54
